# 01 — Load & Preprocess PubMed (Colab)

Runs on Colab. Clones the project from GitHub, downloads a PubMed subset, cleans articles, sentence-tokenizes, and saves JSONL splits to your Google Drive so notebooks 02–05 can reuse them.

**Before running:** `Runtime → Change runtime type → T4 GPU`. (CPU also works for this notebook; GPU just speeds up the dataset cache.)

In [7]:
# 1. Clone the project repo (gets the latest code from GitHub)
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')
print('Working dir:', os.getcwd())

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: '/content/scientific-summarizer'

In [2]:
# 2. Install dependencies
!pip -q install 'transformers>=4.40' 'datasets>=2.18' rouge-score evaluate nltk sentencepiece accelerate tqdm
import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
!nvidia-smi 2>/dev/null || echo 'No GPU (fine for notebook 01).'

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
Mon May 11 19:13:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                    

In [8]:
# 3. Mount Google Drive — used ONLY to persist processed dataset & model checkpoints
#    so notebooks 02-05 don't have to redo this work.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/drive/MyDrive/scientific-summarizer-data'
PROJECT_DIR = DATA_DIR  # alias used by later cells
!mkdir -p {DATA_DIR}/dataset/processed {DATA_DIR}/models {DATA_DIR}/outputs
print('Persistent data dir:', DATA_DIR)

Mounted at /content/drive
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Persistent data dir: /content/drive/MyDrive/scientific-summarizer-data


In [ ]:
# 4. Download PubMed subset
# We use the parquet mirror `ccdv/pubmed-summarization` (same data as
# `scientific_papers/pubmed`, but works with datasets v3+ which dropped
# support for script-based datasets).
from datasets import load_dataset
N_TRAIN, N_VAL, N_TEST = 3000, 400, 400
DATASET = 'ccdv/pubmed-summarization'
train_raw = load_dataset(DATASET, split=f'train[:{N_TRAIN}]')
val_raw   = load_dataset(DATASET, split=f'validation[:{N_VAL}]')
test_raw  = load_dataset(DATASET, split=f'test[:{N_TEST}]')
print(train_raw, val_raw, test_raw)
print('columns:', train_raw.column_names)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ccdv/pubmed-summarization' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ccdv/pubmed-summarization' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ccdv/pubmed-summarization' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`tr

Dataset({
    features: ['article', 'abstract'],
    num_rows: 3000
}) Dataset({
    features: ['article', 'abstract'],
    num_rows: 400
}) Dataset({
    features: ['article', 'abstract'],
    num_rows: 400
})
columns: ['article', 'abstract']


In [10]:
# 5. Quick look at one record
rec = train_raw[0]
print('ARTICLE (first 800 chars):\n', rec['article'][:800])
print('\n--- ABSTRACT ---\n', rec['abstract'][:800])

ARTICLE (first 800 chars):
 a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . 
 anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . 
 snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition program

--- ABSTRACT ---
 background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition stat

In [ ]:
# 6. Clean + sentence-tokenize, filter, save as JSONL
import sys, os
# defensive: re-establish project path so this cell works even if run
# out-of-order or after a kernel restart (avoids ModuleNotFoundError)
if '/content/scientific-summarizer' not in sys.path:
    sys.path.insert(0, '/content/scientific-summarizer')
if os.path.isdir('/content/scientific-summarizer'):
    os.chdir('/content/scientific-summarizer')
import json
from tqdm import tqdm
from preprocessing.clean import clean_article, split_sentences, is_record_valid

OUT = f'{PROJECT_DIR}/dataset/processed'
os.makedirs(OUT, exist_ok=True)

def process_split(ds, out_path):
    kept = dropped = 0
    with open(out_path, 'w', encoding='utf-8') as f:
        for ex in tqdm(ds, desc=os.path.basename(out_path)):
            art = clean_article(ex['article'])
            abs_ = clean_article(ex['abstract'])
            if not is_record_valid(art, abs_):
                dropped += 1; continue
            rec = {
                'input_text': art,
                'target_text': abs_,
                'sentences': split_sentences(art),
            }
            f.write(json.dumps(rec) + '\n')
            kept += 1
    print(f'{out_path}: kept={kept} dropped={dropped}')

process_split(train_raw, f'{OUT}/train.jsonl')
process_split(val_raw,   f'{OUT}/val.jsonl')
process_split(test_raw,  f'{OUT}/test.jsonl')

ModuleNotFoundError: No module named 'preprocessing'

In [ ]:
# 7. Quick stats
import json
import numpy as np

def length_stats(path):
    art_lens, abs_lens, sent_counts = [], [], []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            art_lens.append(len(r['input_text'].split()))
            abs_lens.append(len(r['target_text'].split()))
            sent_counts.append(len(r['sentences']))
    print(path)
    print('  article words   median/p95:', int(np.median(art_lens)), int(np.percentile(art_lens, 95)))
    print('  abstract words  median/p95:', int(np.median(abs_lens)), int(np.percentile(abs_lens, 95)))
    print('  sentences/paper median/p95:', int(np.median(sent_counts)), int(np.percentile(sent_counts, 95)))

length_stats(f'{OUT}/train.jsonl')
length_stats(f'{OUT}/val.jsonl')
length_stats(f'{OUT}/test.jsonl')